# Reference Data Coverage

Read-only analysis of local aircraft, flight, provenance, model, and manufacturer coverage.

In [ ]:
# Load the common database, path, export, and report-header helpers.
%run ../pathutils.ipynb
%run ../database.ipynb
%run ../export.ipynb
%run ../report-header.ipynb

# Keep exports optional and resolve their destination through the shared helper.
export_outputs = True
export_folder = get_export_folder_path()


In [ ]:
# Present consistent report and database metadata before the analysis.
report_metadata = display_report_header('Reference Data Coverage')


In [ ]:
# Load observed aircraft, airlines, and flights with their local-reference coverage flags.
aircraft_coverage = query_data('tracker', construct_query('tracker', 'reports', 'reference-aircraft-coverage.sql', {}))
airline_coverage = query_data('tracker', construct_query('tracker', 'reports', 'reference-airline-coverage.sql', {}))
flight_coverage = query_data('tracker', construct_query('tracker', 'reports', 'reference-flight-coverage.sql', {}))
coverage_summary = query_data('tracker', construct_query('tracker', 'reports', 'reference-coverage-summary.sql', {}))
provenance_summary = query_data('tracker', construct_query('tracker', 'reports', 'reference-provenance-summary.sql', {}))
aircraft_coverage.head(25)


In [ ]:
# Retain the existing chart measures independently from the simplified workbook summary.
coverage_plot_summary = pd.DataFrame({
    'Measure': ['Observed aircraft identified', 'Aircraft observations identified', 'Observed callsigns identified', 'Callsign observations identified'],
    'Coverage %': [
        aircraft_coverage['Aircraft Identified'].mean() * 100,
        aircraft_coverage.loc[aircraft_coverage['Aircraft Identified'] == 1, 'Observations'].sum() / aircraft_coverage['Observations'].sum() * 100,
        flight_coverage.loc[flight_coverage['Callsign'] != 'No callsign', 'Flight Identified'].mean() * 100,
        flight_coverage.loc[flight_coverage['Flight Identified'] == 1, 'Observations'].sum() / flight_coverage['Observations'].sum() * 100,
    ]
}).round(1)
coverage_summary


In [ ]:
# Rank the most valuable aircraft, airline, and flight candidates for local enrichment.
unidentified_aircraft = aircraft_coverage[aircraft_coverage['Aircraft Identified'] == 0].nlargest(25, 'Observations')
unresolved_airlines = airline_coverage[airline_coverage['Airline Identified'] == 0]
unresolved_flights = flight_coverage[(flight_coverage['Flight Identified'] == 0) & (flight_coverage['Callsign'] != 'No callsign')].nlargest(25, 'Observations')
incomplete_aircraft = aircraft_coverage[(aircraft_coverage['Aircraft Identified'] == 1) & ((aircraft_coverage['Model Identified'] == 0) | (aircraft_coverage['Manufacturer Identified'] == 0))].nlargest(25, 'Observations')
display(unidentified_aircraft[['Address', 'Observations', 'Sessions']])
display(unresolved_airlines[['Callsign', 'Observations', 'Sessions']].head(25))
display(unresolved_flights[['Callsign', 'Observations', 'Sessions']])
display(incomplete_aircraft[['Address', 'Registration', 'Model', 'Manufacturer', 'Observations']])


In [ ]:
# Visualise coverage percentages and observation volume by provenance source.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
coverage_plot_summary.plot.bar(ax=axes[0], x='Measure', y='Coverage %', ylim=(0, 100), title='Local reference coverage', legend=False)
provenance_plot = provenance_summary.pivot(index='Provenance Source', columns='Reference Type', values='Observations').fillna(0)
provenance_plot.plot.bar(ax=axes[1], title='Observations by provenance source')
axes[0].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
if export_outputs: export_chart(export_folder, 'reference-data-coverage', 'png')


In [ ]:
# Export coverage details and curation candidates without changing the database.
if export_outputs:
    export_to_spreadsheet(export_folder, 'reference-data-coverage.xlsx', {'Coverage': coverage_summary, 'Aircraft': aircraft_coverage, 'Airlines': airline_coverage, 'Flights': flight_coverage, 'Unresolved Aircraft': unidentified_aircraft, 'Unresolved Airlines': unresolved_airlines, 'Unresolved Flights': unresolved_flights, 'Incomplete Aircraft': incomplete_aircraft, 'Provenance': provenance_summary})
